In [1]:
# Carico la cartella one drive in cui ci sono:
# cartella video nutrition estimation / video annotation nutrition estimation
# cartella video nutrition change / video annotation nutrition change
# cartella video image nutrition estimation/ video annotation image estimation

# come ho scaricato i video?
# partendo dai file json ho lanciato lo script downloader sugli id estratti
# dei video scaricati ho fatto un samplig a N=128 FRAME
# caricato su one drive ed adesso linko da questo notebook
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Verifico scheda video e versione compilatore NVCC disponibile nell'ambiente**
- in questo caso A100 con 40Gb di VRAM

In [2]:
!nvidia-smi
!nvcc --version

Fri Jul 24 17:17:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             53W /  400W |     428MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:

# installo PyTorch 2.7.0
!pip install torch==2.7.0 torchvision==0.22.0 --index-url https://download.pytorch.org/whl/cu128

# installo FlashAttention pre-compilato
#!pip install flash-attn --no-build-isolation --upgrade


Looking in indexes: https://download.pytorch.org/whl/cu128


In [4]:

# Rimuovo la vecchia copia (che ha patch residue) e ri-clono il repo pulito
import os, shutil
if os.path.exists("/content/VideoLLaMA2"):
    shutil.rmtree("/content/VideoLLaMA2")
!git clone https://github.com/DAMO-NLP-SG/VideoLLaMA2.git /content/VideoLLaMA2

import sys
if "/content/VideoLLaMA2" not in sys.path:
    sys.path.insert(0, "/content/VideoLLaMA2")


Cloning into '/content/VideoLLaMA2'...
remote: Enumerating objects: 986, done.
remote: Counting objects: 100% (433/433), done.
remote: Compressing objects: 100% (187/187), done.
remote: Total 986 (delta 365), reused 246 (delta 246), pack-reused 553 (from 2)
Receiving objects: 100% (986/986), 55.81 MiB | 19.42 MiB/s, done.
Resolving deltas: 100% (660/660), done.)


In [5]:
%cd VideoLLaMA2/

/content/VideoLLaMA2


In [6]:
!pip install --upgrade pip setuptools wheel #installo librerie utilitarie per l'utilizzo di librerie pre-compilate

In [7]:
# Versioni ufficiali richieste da VideoLLaMA2 (vedi pyproject.toml del repo):
#   transformers==4.40.0, tokenizers==0.19.1, accelerate==0.26.1, decord==0.6.0
# NB: transformers 4.38.2 era troppo vecchio per le classi custom Videollama2*ForCausalLM.
!pip install "transformers==4.40.0" "tokenizers==0.19.1" "accelerate==0.26.1" \
             peft timm einops "decord==0.6.0" av opencv-python


In [8]:
# NB: NON esiste una env var "TRANSFORMERS_ATTN_IMPLEMENTATION" in transformers.
# L'attention eager va richiesta passando attn_implementation="eager" a model_init
# (vedi cella di caricamento sotto). Questa cella e' lasciata solo come promemoria.
print("Eager attention verra' impostato via attn_implementation='eager' in model_init")


Eager attention verra' impostato via attn_implementation='eager' in model_init


In [9]:
import os
# Evita il lookup del secret HF_TOKEN di Colab che va in timeout e "appende" il download
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"   # download piu' stabile
os.environ.pop("HF_TOKEN", None)

import gc, sys, torch
import pathlib

# Libero la VRAM da tentativi precedenti
for name in ['model', 'processor', 'tokenizer']:
    if name in dir():
        exec(f"del {name}")
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"VRAM libera: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GiB")

# -----------------------------------------------------------------------------
# PATCH encoder.py: il repo forza a mano config._attn_implementation="flash_attention_2"
# sulla CLIP/Siglip vision tower, ma questi modelli non supportano flash-attn -> ValueError.
# Sostituiamo con "eager" DIRETTAMENTE nel file sorgente, prima di importare videollama2.
# -----------------------------------------------------------------------------
_enc = pathlib.Path("/content/VideoLLaMA2/videollama2/model/encoder.py")
_txt = _enc.read_text()
if '"flash_attention_2"' in _txt:
    _enc.write_text(_txt.replace('"flash_attention_2"', '"eager"'))
    print("Patch applicata: flash_attention_2 -> eager in encoder.py")
else:
    print("Nessuna patch necessaria (flash_attention_2 non trovato in encoder.py)")

# Svuoto la cache dei moduli gia' importati, cosi' l'import qui sotto rilegge il file patchato
for _m in list(sys.modules):
    if _m.startswith("videollama2"):
        del sys.modules[_m]

# 1) Scarico lo snapshot a parte, con progresso visibile e resume automatico.
#    (VideoLLaMA2-7B e' pubblico: nessun token necessario.)
from huggingface_hub import snapshot_download
model_path = snapshot_download("DAMO-NLP-SG/VideoLLaMA2-7B", resume_download=True)
print("Modello scaricato in:", model_path)

# 2) Inizializzo passando il path LOCALE (niente altre chiamate di rete).
from videollama2 import model_init as videollama2_model_init

device = "cuda:0"
print("Inizio caricamento modello...")
model, processor, tokenizer = videollama2_model_init(
    model_path,
    device_map={"": device},
    attn_implementation="eager",   # forza eager sul modello linguistico
    use_flash_attn=False,
)
# NB: load_pretrained_model forza internamente torch_dtype=torch.float16,
#     quindi il modello gira in fp16 (va bene su A100). Passare bfloat16 non avrebbe effetto.
model.eval()
print("Modello VideoLLaMA2-7B caricato con successo — dtype:", next(model.parameters()).dtype)
print("num_frames usati dal processor:", getattr(model.config, "num_frames", "default (8)"))


VRAM libera: 38.66 GiB
Patch applicata: flash_attention_2 -> eager in encoder.py


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Modello scaricato in: /root/.cache/huggingface/hub/models--DAMO-NLP-SG--VideoLLaMA2-7B/snapshots/b39f289a67cf0aacd935d646de3bc945e63c3a0f


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Inizio caricamento modello...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Modello VideoLLaMA2-7B caricato con successo — dtype: torch.float16
num_frames usati dal processor: default (8)


In [10]:

import json
import os

# cartella del progetto
BASE_DRIVE_PATH = "/content/drive/MyDrive/Progetto Deep Learning 1"

# Definisco un task come:
# coppia video annotazioni con domande
# in particolare nel dataset abbiamo
# domande sul valore nutrizionale totale su un video / cartella di video 
# domande su come cambia il valore nutrizionale di un video  post inserimento di un ingrediente /cartella di video 
# domande su ingredienti inseriti in un determinato frame di un video  /cartella di video

tasks = {
    "nutrition_estimation": {
        "annotations": os.path.join(BASE_DRIVE_PATH, "nutrition_video_nutrition_estimation.json"),
        "video_dir":   os.path.join(BASE_DRIVE_PATH, "Videos Nutrition Estimation Sampled"),
    },
    "nutrition_change": {
        "annotations": os.path.join(BASE_DRIVE_PATH, "nutrition_nutrition_change.json"),
        "video_dir":   os.path.join(BASE_DRIVE_PATH, "Videos Nutrition Change Sampled"),
    },
    # "image_nutrition_estimation": {.   # da valutare ancora
    #     "annotations": os.path.join(BASE_DRIVE_PATH, "nutrition_image_nutrition_estimation.json"),
    #     "video_dir":   os.path.join(BASE_DRIVE_PATH, "Videos Image Nutrition Estimation Sampled"),
    # },
}


In [11]:

import re
import os


# funzione per costruire un prompt da passare al modello
# diciamo al modello chi è come deve comportarsi, come rispondere etc...
def build_prompt(question, choices):
    choices_text = "\n".join([f"{i}. {c}" for i, c in enumerate(choices)])
    SYSTEM_PROMPT = (
        "You are a precise video analysis assistant. Your task is to track ingredients, "
        "actions, and weight measurements on digital scales to answer nutrition-related questions. "
        "Base your answer ONLY on the visual evidence provided in the video. "
        "CRITICAL: Output ONLY the single digit (0, 1, 2, etc.) corresponding to the correct option. "
        "Do NOT write explanations, do NOT write full sentences, and do NOT add any commentary."
    )
    
    return (
        f"\n{SYSTEM_PROMPT}\n\n"
        f"Question: {question}\n\n"
        f"Choices:\n{choices_text}\n\n"
        f"Reply with ONLY the number (0-{len(choices)-1}) corresponding to the correct answer."
    )


# estrarre l'opzione ritornata dal vlm 
def parse_answer(response_text, num_choices):
    matches = re.findall(r'\b([0-9])\b', response_text)
    for m in reversed(matches):
        idx = int(m)
        if 0 <= idx < num_choices:
            return idx
    return -1



# Esegue inferenza su tutti i video di un task.
# - video_dir: cartella contenente i video campionati
#Restituisce una lista di dizionari con i risultati.
def inference_on_videos(model, annotations, video_dir, processor=None, tokenizer=None):
    
    results = []

    # per ogni annotation item del task lo mando al modello e vedo cosa risponde 
    for key, item in annotations.items():
        video_id   = item["inputs"]["video 1"]["id"] #prendo video in input
        question   = item["question"] #domande in input
        choices    = item["choices"]#possibili risposte
        correct    = item["correct_idx"]    #risposta corretta
        video_path = os.path.join(video_dir, f"{video_id}_sampled.mp4")

        print(f"▶ {key} | video: {video_id}")

        try:
            if not os.path.exists(video_path):
                raise FileNotFoundError(f"Video non trovato: {video_path}")

            from videollama2 import mm_infer   #funzione dentro la repository videollama
            # NB: processor['video'] ricampiona internamente a model.config.num_frames
            #     (8 di default, max 32). NON usa tutti i 128 frame del tuo sampling:
            #     il tuo sampling a monte serve solo a ridurre/uniformare il video sorgente.
            video_tensor = processor['video'](video_path)
            response = mm_infer(
                video_tensor,
                build_prompt(question, choices),
                model,
                tokenizer,
                modal="video",
                do_sample=False,  # greedy decoding, deterministico
                max_new_tokens=64,
            )

            predicted  = parse_answer(response, len(choices))
            is_correct = (predicted == correct)

            print(f"   Risposta grezza: {repr(response[-80:])}")
            print(f"   Predetto: {predicted} | Corretto: {correct} | {'✅' if is_correct else '❌'}\n")

        except Exception as e:
            response   = f"ERRORE: {e}"
            predicted  = -1
            is_correct = False
            print(f"   ⚠️ Errore: {e}\n")

        q_type = question.split("highest")[-1].strip().rstrip("?").strip() if "highest" in question else "unknown"

        # accummulo risultati predetti per poi fare le statistiche
        results.append({
            "key":           key,
            "video_id":      video_id,
            "question":      question,
            "q_type":        q_type,
            "correct_idx":   correct,
            "predicted_idx": predicted,
            "is_correct":    is_correct,
            "raw_response":  response,
        })

    return results


In [12]:

import json
import pandas as pd


# Loop su tutti i task: carica annotation, esegue inferenza, salvo risultati

all_results = []

for task_name, task_cfg in tasks.items():
    print(f"\n{'='*60}")
    print(f"  TASK: {task_name}")
    print(f"{'='*60}")

    with open(task_cfg["annotations"], "r") as f:
        annotations = json.load(f)
    print(f"  Caricate {len(annotations)} annotation")

    task_results = inference_on_videos(
        model,
        annotations,
        video_dir=task_cfg["video_dir"],
        processor=processor,
        tokenizer=tokenizer,
    )

    # Aggiungo la colonna task_name a ogni risultato
    for r in task_results:
        r["task"] = task_name

    all_results.extend(task_results)
    print(f"  Task '{task_name}' completato: {len(task_results)} campioni")

df_all = pd.DataFrame(all_results)
print(f"\nInferenza completata su tutti i task — totale campioni: {len(df_all)}")



  TASK: nutrition_estimation
  Caricate 50 annotation
▶ nutrition_video_nutrition_estimation_0 | video: P03-20240217-131219


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


   Risposta grezza: '3. split red lentils'
   Predetto: 3 | Corretto: 3 | ✅

▶ nutrition_video_nutrition_estimation_1 | video: P03-20240218-190556
   Risposta grezza: '2'
   Predetto: 2 | Corretto: 0 | ❌

▶ nutrition_video_nutrition_estimation_2 | video: P03-20240217-131219
   Risposta grezza: '4'
   Predetto: 4 | Corretto: 4 | ✅

▶ nutrition_video_nutrition_estimation_3 | video: P08-20240614-184945
   Risposta grezza: '4'
   Predetto: 4 | Corretto: 4 | ✅

▶ nutrition_video_nutrition_estimation_4 | video: P03-20240217-131219
   Risposta grezza: '1'
   Predetto: 1 | Corretto: 1 | ✅

▶ nutrition_video_nutrition_estimation_5 | video: P08-20240622-141902
   Risposta grezza: '4'
   Predetto: 4 | Corretto: 4 | ✅

▶ nutrition_video_nutrition_estimation_6 | video: P08-20240617-130401
   Risposta grezza: '2'
   Predetto: 2 | Corretto: 2 | ✅

▶ nutrition_video_nutrition_estimation_7 | video: P03-20240217-131219
   Risposta grezza: '3'
   Predetto: 3 | Corretto: 3 | ✅

▶ nutrition_video_nutrition

In [13]:

# ── Analisi delle performance globali ─────────────────────────────────────────

df = df_all.copy()

total     = len(df)
correct   = df["is_correct"].sum()
accuracy  = correct / total * 100
random_baseline = 100 / 5  # 5 scelte sempre

print("=" * 60)
print(f"  ACCURACY GLOBALE  : {accuracy:.1f}%  ({correct}/{total})")
print(f"  Baseline random   : {random_baseline:.1f}%")
print("=" * 60)

# Accuracy per task
print("\n Accuracy per task:")
acc_by_task = (
    df.groupby("task")["is_correct"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "corrette", "count": "totali"})
)
acc_by_task["accuracy_%"] = (acc_by_task["corrette"] / acc_by_task["totali"] * 100).round(1)
print(acc_by_task.to_string())

# Accuracy per tipo di domanda
print("\n Accuracy per tipo di domanda:")
acc_by_type = (
    df.groupby("q_type")["is_correct"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "corrette", "count": "totali"})
)
acc_by_type["accuracy_%"] = (acc_by_type["corrette"] / acc_by_type["totali"] * 100).round(1)
print(acc_by_type.to_string())

# Accuracy per video
print("\n Accuracy per video:")
acc_by_video = (
    df.groupby("video_id")["is_correct"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "corrette", "count": "totali"})
)
acc_by_video["accuracy_%"] = (acc_by_video["corrette"] / acc_by_video["totali"] * 100).round(1)
print(acc_by_video.to_string())

# Salva i risultati su CSV nel Drive
output_csv = "/content/drive/MyDrive/inference_results_all_tasks.csv"
df.to_csv(output_csv, index=False)
print(f"\n💾 Risultati salvati in: {output_csv}")


  ACCURACY GLOBALE  : 32.0%  (32/100)
  Baseline random   : 20.0%

 Accuracy per task:
                      corrette  totali  accuracy_%
task                                              
nutrition_change             5      50        10.0
nutrition_estimation        27      50        54.0

 Accuracy per tipo di domanda:
                         corrette  totali  accuracy_%
q_type                                               
calories in this recipe         5      14        35.7
carbs in this recipe            9      13        69.2
fat in this recipe              6      10        60.0
protein in this recipe          7      13        53.8
unknown                         5      50        10.0

 Accuracy per video:
                     corrette  totali  accuracy_%
video_id                                         
P01-20240203-130505         0       2         0.0
P01-20240203-135502         0       3         0.0
P01-20240204-121042         0       2         0.0
P03-20240217-131219        

## Inferenza con Video Panels

Invece di passare il video `.mp4` al modello, passiamo i **frame pannellati** (immagini JPEG 2×2) prodotti dallo script `mosaic.py`.

Il processor di VideoLLaMA2 campiona **`num_frames` panel** (8 di default, max 32 = `MAX_FRAMES`). Poiché ogni panel contiene 4 frame in griglia 2×2, con 8 panel il modello "vede" fino a **32 frame originali** a parità di finestra di contesto → copertura temporale ~4× maggiore rispetto agli 8 frame del video liscio.

> Se vuoi che il modello usi più panel (es. 16), va aumentato `model.config.num_frames` **prima** di `model_init` (il processor eredita quel valore).


In [20]:
import json
import os
import re
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm

# =====================================================================
# CONFIGURAZIONE PANELS
# =====================================================================
BASE_DRIVE_PATH = "/content/drive/MyDrive/Progetto Deep Learning 1"

# Task con le cartelle dei panels (output di mosaic.py)
tasks_panels = {
    "nutrition_estimation": {
        "annotations": os.path.join(BASE_DRIVE_PATH, "nutrition_video_nutrition_estimation.json"),
        "panels_dir":  os.path.join(BASE_DRIVE_PATH, "videos_image_nutrition_estimation_panels"),
    },
    "nutrition_change": {
        "annotations": os.path.join(BASE_DRIVE_PATH, "nutrition_nutrition_change.json"),
        "panels_dir":  os.path.join(BASE_DRIVE_PATH, "videos_nutrition_change_panels"),
    },
    # "image_nutrition_estimation": {
    #     "annotations": os.path.join(BASE_DRIVE_PATH, "nutrition_image_nutrition_estimation.json"),
    #     "panels_dir":  os.path.join(BASE_DRIVE_PATH, "videos_image_nutrition_estimation_panels"),
    # },
}


In [21]:
def load_panels_as_video_tensor(panels_dir, video_id, processor):
    """
    Carica i panel JPEG di un video e li passa al PROCESSOR NATIVO di VideoLLaMA2.

    IMPORTANTE: non si costruisce il tensore a mano. Il process_video di VideoLLaMA2
    accetta gia' una lista di path immagine (o una cartella) e applica il preprocessing
    corretto per il visual encoder CLIP:
        - resize a 336x336 (NON 384),
        - normalizzazione CLIP (mean=[0.481,0.458,0.408], std=[0.269,0.261,0.276]) — NON ImageNet,
        - padding "expand2square",
        - campionamento uniforme a model.config.num_frames (8) e cap a MAX_FRAMES (32).
    Costruire il tensore a mano con norm/size sbagliati produce output spazzatura.
    """
    video_panel_dir = os.path.join(panels_dir, video_id)
    if not os.path.exists(video_panel_dir):
        raise FileNotFoundError(f"Cartella panels non trovata: {video_panel_dir}")

    # Lista ordinata dei panel (i nomi sono zero-padded: panel_0000.jpg, panel_0001.jpg, ...)
    panel_files = sorted([
        os.path.join(video_panel_dir, f)
        for f in os.listdir(video_panel_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])
    if len(panel_files) == 0:
        raise FileNotFoundError(f"Nessun panel trovato in: {video_panel_dir}")

    # process_video gestisce lista di path -> tensore (T, C, H, W) pronto per mm_infer
    video_tensor = processor['video'](panel_files)
    return video_tensor


def inference_on_panels(model, annotations, panels_dir, processor, tokenizer):
    """
    Esegue inferenza usando i frame pannellati invece dei video originali.
    Ogni panel è una griglia 2x2 che contiene 4 frame del video.
    """
    results = []
    
    for key, item in annotations.items():
        video_id   = item["inputs"]["video 1"]["id"]
        question   = item["question"]
        choices    = item["choices"]
        correct    = item["correct_idx"]
        
        print(f"▶ {key} | video: {video_id} (panels)")
        
        try:
            video_tensor = load_panels_as_video_tensor(panels_dir, video_id, processor)
            
            from videollama2 import mm_infer
            response = mm_infer(
                video_tensor,
                build_prompt(question, choices),
                model,
                tokenizer,
                modal="video",
                do_sample=False,
                max_new_tokens=64,
            )
            
            predicted  = parse_answer(response, len(choices))
            is_correct = (predicted == correct)
            
            print(f"   Risposta grezza: {repr(response[-80:])}")
            print(f"   Predetto: {predicted} | Corretto: {correct} | {'✅' if is_correct else '❌'}\n")
        
        except Exception as e:
            response   = f"ERRORE: {e}"
            predicted  = -1
            is_correct = False
            print(f"   ⚠️ Errore: {e}\n")
        
        q_type = question.split("highest")[-1].strip().rstrip("?").strip() if "highest" in question else "unknown"
        
        results.append({
            "key":           key,
            "video_id":      video_id,
            "question":      question,
            "q_type":        q_type,
            "correct_idx":   correct,
            "predicted_idx": predicted,
            "is_correct":    is_correct,
            "raw_response":  response,
        })
    
    return results


In [22]:
# =====================================================================
# ESEGUO INFERENZA CON PANELS SU TUTTI I TASK
# =====================================================================

import json
import pandas as pd

all_results_panels = []

for task_name, task_cfg in tasks_panels.items():
    print(f"\n{'='*60}")
    print(f"  TASK (PANELS): {task_name}")
    print(f"{'='*60}")
    
    if not os.path.exists(task_cfg["annotations"]):
        print(f"  ⚠️ Annotations non trovate: {task_cfg['annotations']}, skip.")
        continue
    if not os.path.exists(task_cfg["panels_dir"]):
        print(f"  ⚠️ Panels dir non trovata: {task_cfg['panels_dir']}, skip.")
        continue
    
    with open(task_cfg["annotations"], "r") as f:
        annotations = json.load(f)
    print(f"  Caricate {len(annotations)} annotation")
    
    task_results = inference_on_panels(
        model,
        annotations,
        panels_dir=task_cfg["panels_dir"],
        processor=processor,
        tokenizer=tokenizer,
    )
    
    for r in task_results:
        r["task"] = task_name
        r["method"] = "video_panels_2x2"
    
    all_results_panels.extend(task_results)
    print(f"  Task '{task_name}' completato: {len(task_results)} campioni")

df_panels = pd.DataFrame(all_results_panels)
print(f"\nInferenza PANELS completata — totale campioni: {len(df_panels)}")



  TASK (PANELS): nutrition_estimation
  Caricate 50 annotation
▶ nutrition_video_nutrition_estimation_0 | video: P03-20240217-131219 (panels)
   Risposta grezza: '3. split red lentils'
   Predetto: 3 | Corretto: 3 | ✅

▶ nutrition_video_nutrition_estimation_1 | video: P03-20240218-190556 (panels)
   Risposta grezza: '2'
   Predetto: 2 | Corretto: 0 | ❌

▶ nutrition_video_nutrition_estimation_2 | video: P03-20240217-131219 (panels)
   Risposta grezza: '4'
   Predetto: 4 | Corretto: 4 | ✅

▶ nutrition_video_nutrition_estimation_3 | video: P08-20240614-184945 (panels)
   Risposta grezza: '4'
   Predetto: 4 | Corretto: 4 | ✅

▶ nutrition_video_nutrition_estimation_4 | video: P03-20240217-131219 (panels)
   Risposta grezza: '1'
   Predetto: 1 | Corretto: 1 | ✅

▶ nutrition_video_nutrition_estimation_5 | video: P08-20240622-141902 (panels)
   Risposta grezza: '4'
   Predetto: 4 | Corretto: 4 | ✅

▶ nutrition_video_nutrition_estimation_6 | video: P08-20240617-130401 (panels)
   Risposta grez

In [23]:
# =====================================================================
# PERFORMANCE PANELS vs BASELINE (video originali)
# =====================================================================

df_p = df_panels.copy()

total     = len(df_p)
correct   = df_p["is_correct"].sum()
accuracy  = correct / total * 100
random_baseline = 100 / 5

print("=" * 60)
print(f"  VIDEO PANELS (2×2) — ACCURACY GLOBALE: {accuracy:.1f}%  ({correct}/{total})")
print(f"  Baseline random: {random_baseline:.1f}%")
print("=" * 60)

# Accuracy per task
print("\n📊 Accuracy per task (panels):")
acc_by_task = (
    df_p.groupby("task")["is_correct"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "corrette", "count": "totali"})
)
acc_by_task["accuracy_%"] = (acc_by_task["corrette"] / acc_by_task["totali"] * 100).round(1)
print(acc_by_task.to_string())

# Accuracy per tipo di domanda
print("\n📊 Accuracy per tipo di domanda (panels):")
acc_by_type = (
    df_p.groupby("q_type")["is_correct"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "corrette", "count": "totali"})
)
acc_by_type["accuracy_%"] = (acc_by_type["corrette"] / acc_by_type["totali"] * 100).round(1)
print(acc_by_type.to_string())

# ── Confronto con baseline (se df_all esiste dalla sezione precedente) ──
try:
    df_base = df_all.copy()
    print("\n" + "=" * 60)
    print("  CONFRONTO: Video Originali vs Video Panels")
    print("=" * 60)
    
    base_acc = df_base["is_correct"].mean() * 100
    panel_acc = df_p["is_correct"].mean() * 100
    delta = panel_acc - base_acc
    
    print(f"  Video originali (sampling):  {base_acc:.1f}%")
    print(f"  Video Panels (2×2):          {panel_acc:.1f}%")
    print(f"  Delta:                       {'+' if delta >= 0 else ''}{delta:.1f}%")
    
    # Confronto per task
    print("\n  Per task:")
    for task in df_p["task"].unique():
        base_t = df_base[df_base["task"] == task]["is_correct"].mean() * 100 if task in df_base["task"].values else float('nan')
        panel_t = df_p[df_p["task"] == task]["is_correct"].mean() * 100
        d = panel_t - base_t if not np.isnan(base_t) else float('nan')
        print(f"    {task:30s}  base={base_t:5.1f}%  panels={panel_t:5.1f}%  Δ={d:+.1f}%")

except NameError:
    print("\n⚠️ df_all non trovato — esegui prima la sezione di inferenza baseline per il confronto.")

# Salva risultati panels
output_csv_panels = "/content/drive/MyDrive/inference_results_panels.csv"
df_p.to_csv(output_csv_panels, index=False)
print(f"\n💾 Risultati panels salvati in: {output_csv_panels}")


  VIDEO PANELS (2×2) — ACCURACY GLOBALE: 32.0%  (32/100)
  Baseline random: 20.0%

📊 Accuracy per task (panels):
                      corrette  totali  accuracy_%
task                                              
nutrition_change            11      50        22.0
nutrition_estimation        21      50        42.0

📊 Accuracy per tipo di domanda (panels):
                         corrette  totali  accuracy_%
q_type                                               
calories in this recipe         4      14        28.6
carbs in this recipe            9      13        69.2
fat in this recipe              2      10        20.0
protein in this recipe          6      13        46.2
unknown                        11      50        22.0

  CONFRONTO: Video Originali vs Video Panels
  Video originali (sampling):  32.0%
  Video Panels (2×2):          32.0%
  Delta:                       +0.0%

  Per task:
    nutrition_estimation            base= 54.0%  panels= 42.0%  Δ=-12.0%
    nutrition_change